In [ ]:
import pubchempy as pcp, pandas as pd, sys, re, requests, math, numpy as np

In [ ]:
# Set input/output file names here
INPUT_FILE = 'PubChem_input1.xlsx'
OUTPUT_FILE = 'PubChem_output1.xlsx'

In [ ]:
# Read in dataframe from excel
t = pd.read_excel(INPUT_FILE)
t.head(50)

In [ ]:
# Put columns in list
cas_list = t['CAS#'].tolist()
name_list = t['Name'].tolist()

In [ ]:
# Search names using pubchempy and get a list of CIDs
cid_lst = []

for x in name_list:
    cids = pcp.get_cids(x, 'name')
    if len(cids) > 0:
        cid = cids[0]
    else:
        cid = 0
    cid_lst.append(cid)

In [ ]:
# Search CAS numbers as names using pubchempy and get list of CIDs
cas_cids = []

for x in cas_list:
    if str(x) != 'nan':
        cids = pcp.get_cids(str(x), 'name')
        if len(cids) > 0:
            cid = cids[0]
        else:
            x = 'CAS-' + x
            cids = pcp.get_cids(str(x), 'name')
            if len(cids) > 0:
                cid = cids[0]
            else:
                cid = 0
    else:
        cid = 0
    cas_cids.append(cid)

In [ ]:
df = pd.DataFrame()
df['Labeled Names'] = name_list
df['Labeled CAS'] = cas_list
df['CID from Name'] = cid_lst
df['CID from CAS #'] = cas_cids

In [ ]:
df.head(50)

In [ ]:
# Function to extract data from PubChem JSON response
def get_pc_data(cid):
    pc_cas = []
    other_cas = []
    rel_cas = []
    syn_lst = []
    ghs_pict = []
    ghs_haz = []

    resp = requests.get('https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/' + str(cid) + '/JSON')
    data = resp.json()

    pc_name = data['Record']['RecordTitle']

    for a in data['Record']['Section']:
        if a['TOCHeading'] == 'Names and Identifiers':
            for b in a['Section']:
                if b['TOCHeading'] == 'Other Identifiers':
                    for c in b['Section']:
                        if c['TOCHeading'] == 'CAS':
                            pc_cas = c['Information'][0]['Value']['StringWithMarkup'][0]['String']
                            for d in c['Information']:
                                if 'Name' in d.keys():
                                    if d['Name'] == 'Other CAS':
                                        other_cas.append(d['Value']['StringWithMarkup'][0]['String'])
                                    elif d['Name'] == 'Related CAS':
                                        rel_cas.append(d['Value']['StringWithMarkup'][0]['String'])
                                else:
                                    if (d['Value']['StringWithMarkup'][0]['String']) != pc_cas:
                                        other_cas.append(d['Value']['StringWithMarkup'][0]['String'])

                elif b['TOCHeading'] == 'Computed Descriptors':
                    for c in b['Section']:
                        if c['TOCHeading'] == 'IUPAC Name':
                            iupac = c['Information'][0]['Value']['StringWithMarkup'][0]['String']

                elif b['TOCHeading'] == 'Synonyms':
                    for c in b['Section']:
                        if c['TOCHeading'] == 'Depositor-Supplied Synonyms':
                            d = (c['Information'][0]['Value']['StringWithMarkup'])
                            for e in d:
                                syn_lst.append(e['String'])

        elif a['TOCHeading'] == 'Safety and Hazards':
            for b in a['Section']:
                if b['TOCHeading'] == 'Hazards Identification':
                    for c in b['Section']:
                        if c['TOCHeading'] == 'GHS Classification':
                            for d in c['Information']:
                                if 'Name' in d.keys():
                                    if d['Name'] == 'Pictogram(s)':
                                        for e in (d['Value']['StringWithMarkup'][0]['Markup']):
                                            ghs_pict.append(e['Extra'])
                                    elif d['Name'] == 'GHS Hazard Statements':
                                        for e in d['Value']['StringWithMarkup']:
                                            ghs_haz.append(e['String'])

    other_cas = list(set(other_cas))
    if len(other_cas) == 0:
        other_cas = 'None'
    elif len(other_cas) == 1:
        other_cas = other_cas[0]
    else:
        other_cas = '\n'.join(other_cas)

    rel_cas = list(set(rel_cas))
    if len(rel_cas) == 0:
        rel_cas = 'None'
    elif len(rel_cas) == 1:
        rel_cas = rel_cas[0]
    else:
        rel_cas = '\n'.join(rel_cas)

    new_syn_lst = []
    marker = set()
    for x in syn_lst:
        xx = x.lower()
        if xx not in marker:
            marker.add(xx)
            new_syn_lst.append(x)

    if len(new_syn_lst) == 0:
        new_syn_lst = 'None'
    elif len(new_syn_lst) == 1:
        new_syn_lst = new_syn_lst[0]
    else:
        new_syn_lst = '\n'.join(new_syn_lst[:5])

    ghs_pict = list(set(ghs_pict))
    if len(ghs_pict) == 0:
        ghs_pict = 'None'
    elif len(ghs_pict) == 1:
        ghs_pict = ghs_pict[0]
    else:
        ghs_pict = '\n'.join(ghs_pict)

    if len(ghs_haz) == 0:
        ghs_haz = 'None'
    else:
        ghs_haz = [x for x in ghs_haz if len(x) > 0]
        ghs_haz = [x for x in ghs_haz if x[0] == 'H']
        ghs_haz = list(set(ghs_haz))
        ghs_haz = '\n'.join(ghs_haz)

    return pc_name, pc_cas, other_cas, rel_cas, new_syn_lst, ghs_pict, ghs_haz

In [ ]:
# Extract data and add to lists
pubchem_names = []
pubchem_cas = []
other_cas_numbers = []
rel_cas_numbers = []
synonyms = []
ghs_pictograms = []
ghs_hazards = []

for x in df['CID from CAS #'].tolist():
    if x != 0:
        pc_name, pc_cas, other_cas, rel_cas, new_syn_lst, ghs_pict, ghs_haz = get_pc_data(x)
        pubchem_names.append(pc_name)
        pubchem_cas.append(pc_cas)
        other_cas_numbers.append(other_cas)
        rel_cas_numbers.append(rel_cas)
        synonyms.append(new_syn_lst)
        ghs_pictograms.append(ghs_pict)
        ghs_hazards.append(ghs_haz)
    else:
        pubchem_names.append('Not Found')
        pubchem_cas.append('nan')
        other_cas_numbers.append('nan')
        rel_cas_numbers.append('nan')
        synonyms.append('nan')
        ghs_pictograms.append('nan')
        ghs_hazards.append('nan')

In [ ]:
# Add results to dataframe
df['PubChem Name'] = pubchem_names
df['PubChem CAS #'] = pubchem_cas
df['Other CAS #s'] = other_cas_numbers
df['Related CAS #s'] = rel_cas_numbers
df['Top 5 Chem Names'] = synonyms
df['GHS Pictogram Names'] = ghs_pictograms
df['GHS Hazard Statements'] = ghs_hazards

In [ ]:
df.head(50)

In [ ]:
# Write results to Excel
with pd.ExcelWriter(OUTPUT_FILE) as writer:
    df.to_excel(writer, index=False)